In [ ]:
from elevant import settings
from elevant.linkers.linking_system import LinkingSystem

linker = LinkingSystem(
    linker_name="graph-llm",
    config_path='/media/volume/LLMRag2/.local/ActDiseaseEL/configs/graph-llm.config.json',
    coref_linker=None,
    min_score=0,
    type_mapping_file=settings.QID_TO_WHITELIST_TYPES_DB,
    custom_kb=True)

text = "W. C. HUEPER (Arch. Intern. Med., December, 1928, p. 893) recalls that Schultz, in 1922, described a type of necrotic angina accompanied by a marked absolute and especially granulocytic leucopenia, and regarded these symptoms as manifestations of a disease which he called \"agranulocytosis.\" Since then about 125 cases have been recorded under this name or that of \"agranulocytic angina\" (Friedemann). Hueper now records observations on five cases seen between November, 1927, and April, 1928. The etiology is unknown. Most authorities regard it as an infectious disease represented by a septicaemia with an atypical reaction of the haemopoietic system, due either to bacteria with a special affinity and toxicity to the granulocytic system or to an atrophy and aplasia of this organ caused by septic infection. According to these investigations he had a case of a severe non septic septic or specific (septicocele, B. pyogenicus, or fusosporilis). The disease starts after a period of prolonged ill health, cr, more frequently, in previously healthy subjects, with high continued fever, malaise, dyspnoea, and dyspnoea; slight injuries is present in about 50 per cent. The patient rapidly gets worse, and death occurs after coma of two to seven days' duration; there may be rarely remissions of a few days to several weeks. The outcome is usually fatal. At the onset the tonsils are enlarged and reduced, and show yellowish white plugs, which merge to form dirty grey or yellowish coats; on removal of these an ulcerated surface appears. Sloughing of the tonsils rapidly ensues, and a similar necrotic process may be found on the pharynx, uvula, palate, tongue, pharynx, gums, anus, vulva, vagina, and cervix. Staphylococcus, streptococci, and pneumococci are the organisms usually found in the throat. The blood shows a considerable leucopenia; the granulocytic cells decrease first and may disappear completely, but there is also a lymphocytic diminution, which slowly follows that of the granulocytic cells. The red cells are normal or exhibit only slight changes. Blood cultures are positive in only 10 per cent., when they contain haemolytic and non haemolytic streptococci, Streptococcus viridans, B. pyogenicus, B. cidii lactis, and B. coli. The disease is at far more frequent in women than in men and mainly affects the middle aged; it is apparently non contagious. The prognosis is bad, but not absolutely hopeless. Recovery has followed the use of anti streptococcus serum and z ray applications to the long bones."
out = linker.linker._detect_entities_with_llm(text)
nout = linker.linker._get_candidates_for_entities(out)
nout = linker.linker._get_candidates_for_entities(nout)
print(out)
print(nout)

/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 5/5 [00:18<00:00,  3.67s/it]


Span dictionary (33): {(173, 197): {'span_text': 'granulocytic leucopenia,', 'start_char': 173, 'end_char': 197, 'entities': [{'synonyms': ['granulocytopenia', 'Granulocytopenic disorder', 'Granulopenia'], 'xrefs': ['ICD10CM:D70', 'MESH:D000380', 'SNOMEDCT_US_2023_03_01:154830007', 'UMLS_CUI:C0001824'], 'id': 'DOID:12987', 'name': 'agranulocytosis', 'def': 'A leukopenia that is characterized by a severe lack of of granulocytes with a drop in granulocyte concentration below 200 cells/mm³ of blood


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Confirmed entities: 0
Unconfirmed entities: 31


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [1]:
from elevant.models.entity_database import EntityDatabase

entity_db = EntityDatabase()
entity_db.load_entity_names()
entity_db.load_alias_to_entities()
entity_db.load_hyperlink_to_most_popular_candidates()
entity_db.load_sitelink_counts()

In [11]:
None or 1

1

In [ ]:
import os
import multiprocessing as mp
import requests
import time
import subprocess
import psutil
import json
from typing import List, Optional, Dict, Any
from transformers import AutoTokenizer

# Set multiprocessing method before any imports
os.environ['VLLM_ENGINE_MP_START_METHOD'] = 'spawn'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

if mp.get_start_method(allow_none=True) != 'spawn':
    mp.set_start_method('spawn', force=True)

DEFAULT_VLLM_PORT = 8000
DEFAULT_HOST = "127.0.0.1"
VLLM_SERVER_PROCESS = None

class LLMClient:
    def __init__(self, model_name: str, use_4bit: bool = True, port: int = None):
        self.model_name = model_name
        self.port = port or int(os.getenv('VLLM_PORT', DEFAULT_VLLM_PORT))
        self.host = os.getenv('VLLM_HOST', DEFAULT_HOST)
        self.base_url = f"http://{self.host}:{self.port}"
        self.server_started = False
        
        # Load tokenizer for chat template support
        print(f"Loading tokenizer for {model_name}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True
            )
            self.has_chat_template = (
                hasattr(self.tokenizer, 'chat_template') and 
                self.tokenizer.chat_template is not None
            )
            if self.has_chat_template:
                print(f"✅ Chat template detected. Will auto-format user prompts.")
            else:
                print(f"⚠️ No chat template found. Using raw prompts.")
        except Exception as e:
            print(f"⚠️ Failed to load tokenizer: {e}. Using raw prompts.")
            self.tokenizer = None
            self.has_chat_template = False
        
        # Check if server is running
        if not self._is_server_running():
            print(f"🚀 No vLLM server found. Starting on {self.host}:{self.port}...")
            self._start_server()
            self.server_started = True
        else:
            print(f"✅ Connected to existing vLLM server at {self.host}:{self.port}")
        
        # Verify model is loaded
        if not self._verify_model():
            raise RuntimeError(f"Model {model_name} not loaded on server")
    
    def _is_server_running(self) -> bool:
        """Check if vLLM server is responding"""
        try:
            response = requests.get(f"{self.base_url}/health", timeout=2)
            return response.status_code == 200
        except requests.exceptions.RequestException:
            return False
    
    def _verify_model(self) -> bool:
        """Verify the correct model is loaded"""
        try:
            response = requests.get(f"{self.base_url}/v1/models", timeout=5)
            if response.status_code == 200:
                models = response.json().get("data", [])
                for model in models:
                    if model.get("id") == self.model_name:
                        return True
            return False
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Cannot verify model: {e}")
            return False
    
    def _start_server(self):
        """Launch vLLM server as background process"""
        # Build command
        cmd = [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            "--model", self.model_name,
            "--host", self.host,
            "--port", str(self.port),
            "--tensor-parallel-size", "1",
            "--dtype", "float16",
            "--trust-remote-code"
        ]
        
        if "AWQ" in self.model_name:
            cmd.extend(["--quantization", "AWQ"])
        
        # Start server in background
        print(f"Running: {' '.join(cmd)}")
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            start_new_session=True
        )
        
        # Wait for server to be ready
        max_retries = 60
        for i in range(max_retries):
            if self._is_server_running() and self._verify_model():
                print("✅ Server ready!")
                return
            time.sleep(1)
            if i % 10 == 0:
                print(f"⏳ Waiting for server to start... ({i}s)")
        
        proc.terminate()
        raise RuntimeError("Failed to start vLLM server within timeout")
    
    def _apply_chat_template(self, prompt: str) -> str:
        if not self.has_chat_template:
            return prompt
        
        try:
            return self.tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception as e:
            print(f"⚠️ Failed to apply chat template: {e}. Using raw prompt.")
            return prompt

    def call_batch(
        self,
        prompts: List[str],
        max_new_tokens: int = 1024,
        temperature: float = 0.1,
        **sampling_kwargs
    ) -> List[str]:
        if not prompts: return []
        
        formatted_prompts = [self._apply_chat_template(p) for p in prompts]
        
        try:
            payload = {
                "model": self.model_name,
                "prompt": formatted_prompts,
                "max_tokens": max_new_tokens,
                "temperature": temperature,
                **sampling_kwargs
            }
            
            response = requests.post(
                f"{self.base_url}/v1/completions",
                json=payload
            )
            response.raise_for_status()
            
            results = []
            for choice in response.json()["choices"]:
                text = choice["text"].strip()
                if '</think>' in text:
                    text = text.split('</think>')[-1].strip()
                results.append(text)
            
            return results
            
        except requests.exceptions.RequestException as e:
            print(f"⚠️ API call failed: {e}")
            return [""] * len(prompts)
        except Exception as e:
            print(f"⚠️ Unexpected error: {e}")
            return [""] * len(prompts)

    def call(self, prompt: str, **kwargs) -> str:
        return self.call_batch([prompt], **kwargs)[0]
    
    def shutdown(self):
        if hasattr(self, 'server_started') and self.server_started:
            try: requests.post(f"{self.base_url}/shutdown", timeout=2)
            except: pass


def get_shared_llm_client(model_name: str = None, port: int = None) -> LLMClient:
    """Get or create shared vLLM client"""
    if model_name is None:
        model_name = os.getenv('VLLM_MODEL', 'Orion-zhen/Qwen3-8B-AWQ')
    
    return LLMClient(model_name=model_name, port=port)

client1 = get_shared_llm_client()


/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading tokenizer for Orion-zhen/Qwen3-8B-AWQ...
✅ Chat template detected. Will auto-format user prompts.
✅ Connected to existing vLLM server at 127.0.0.1:8000


In [17]:
# client1.base_url
requests.post(f"{client1.base_url}/shutdown", timeout=2)

<Response [404]>